# Notebook 03 — Weather-Aware U-Net (Multimodal) · v2
**Model B: Multi-scale FiLM U-Net with CBAM attention + improved training**

Key upgrades over v1:
- Multi-scale FiLM weather fusion at ALL 4 decoder levels (not just bottleneck)
- CBAM attention on every skip connection
- Deeper weather MLP with LayerNorm + GELU
- DropBlock bottleneck regularisation
- AdamW + cosine annealing warm restarts
- SymmetricUnifiedFocalLoss
- Test-time augmentation (TTA) for evaluation

In [ ]:
import sys, os
os.chdir(r'd:\flood_segmentation')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/flood_segmentation
    %pip install -q rasterio tqdm scikit-learn scipy

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

sys.path.insert(0, os.path.abspath('.'))
from src.datasets import build_datasets, WEATHER_FEATURES
from src.models   import get_model
from src.train    import train_model, evaluate_model
from src.utils    import visualize_predictions, plot_training_history

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Hyperparameters (keep same as Notebook 02 for fair comparison) ───────────
IMG_SIZE    = 256
BATCH_SIZE  = 4      # physical batch; effective = BATCH_SIZE * ACCUM_STEPS = 8
ACCUM_STEPS = 2
NUM_EPOCHS  = 60
LR          = 3e-4
PATIENCE    = 15
WARMUP      = 5
COSINE_T0   = 20
NUM_WORKERS = 0
N_WEATHER   = len(WEATHER_FEATURES)   # 5 features

DATA_DIR    = 'data/raw'
WEATHER_CSV = 'data/processed/weather.csv'
SAVE_PATH   = 'results/saved_models/multimodal_unet_v2.pth'
FIG_DIR     = 'results/figures'
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Weather features ({N_WEATHER}): {WEATHER_FEATURES}')

In [ ]:
# --- Build datasets (same seed = same split as Notebook 02) ---
train_ds, val_ds, test_ds = build_datasets(
    data_dir=DATA_DIR,
    weather_csv_path=WEATHER_CSV,
    img_size=IMG_SIZE,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=42,   # CRITICAL: must match Notebook 02
)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

# Inspect a weather vector
img, weather, mask = train_ds[0]
print(f'\nWeather vector (normalised): {weather.numpy().round(4)}')
for feat, val in zip(WEATHER_FEATURES, weather.numpy()):
    print(f'  {feat:15s}: {val:.4f}')

In [ ]:
# --- Build Model B: Weather-Aware U-Net v2 (multi-scale FiLM + CBAM) ---
model_B = get_model('multimodal', img_ch=2, weather_dim=N_WEATHER,
                    base_ch=64, drop_prob=0.1)
n_params = sum(p.numel() for p in model_B.parameters() if p.requires_grad)
print(f'WeatherAwareUNet (v2 multi-scale FiLM + CBAM) — Trainable parameters: {n_params:,}')

# Test forward pass
model_B.eval()
with torch.no_grad():
    dummy_img = torch.rand(2, 2, IMG_SIZE, IMG_SIZE)
    dummy_w   = torch.rand(2, N_WEATHER)
    out = model_B(dummy_img, dummy_w)
print(f'Test output shape: {out.shape}  (raw logits — no sigmoid)')

In [ ]:
# --- Train ---
history_B, best_iou_B = train_model(
    model=model_B,
    train_dataset=train_ds,
    val_dataset=val_ds,
    is_multimodal=True,     # ← key difference
    save_path=SAVE_PATH,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    device=device,
    warmup_epochs=WARMUP,
    accum_steps=ACCUM_STEPS,
    cosine_T0=COSINE_T0,
)
print(f'\nBest Val IoU (Weather-Aware U-Net v2): {best_iou_B:.4f}')

In [ ]:
# --- Training curves ---
fig = plot_training_history(
    history_B,
    save_path=f'{FIG_DIR}/03_multimodal_v2_history.png',
    title='Weather-Aware U-Net v2 — Training History',
)
plt.show()

In [ ]:
# --- Evaluate on test set (with TTA) ---
test_metrics_B = evaluate_model(
    model=get_model('multimodal', img_ch=2, weather_dim=N_WEATHER,
                    base_ch=64, drop_prob=0.1),
    test_dataset=test_ds,
    checkpoint_path=SAVE_PATH,
    is_multimodal=True,
    batch_size=BATCH_SIZE,
    device=device,
    use_tta=True,
)

os.makedirs('results', exist_ok=True)
with open('results/metrics_multimodal.json', 'w') as f:
    json.dump(test_metrics_B, f, indent=2)
print('Test metrics saved to results/metrics_multimodal.json')

In [ ]:
# --- Compare with Baseline ---
with open('results/metrics_baseline.json') as f:
    test_metrics_A = json.load(f)

metrics = ['iou', 'dice', 'precision', 'recall', 'f1', 'loss']
comparison_df = pd.DataFrame({
    'Baseline U-Net v2':      {m: test_metrics_A.get(m, float('nan')) for m in metrics},
    'Weather-Aware UNet v2':  {m: test_metrics_B.get(m, float('nan')) for m in metrics},
}).T
print('\n── Test Set Comparison ──────────────────────────────────────')
print(comparison_df.round(4).to_string())
delta_iou = test_metrics_B['iou'] - test_metrics_A['iou']
print(f'\nΔ IoU (Multimodal − Baseline): {delta_iou:+.4f}')

In [ ]:
# --- Qualitative predictions ---
from torch.utils.data import DataLoader

model_B_loaded = get_model('multimodal', img_ch=2, weather_dim=N_WEATHER,
                           base_ch=64, drop_prob=0.1)
ckpt = torch.load(SAVE_PATH, map_location=device)
model_B_loaded.load_state_dict(ckpt['model_state_dict'])
model_B_loaded = model_B_loaded.to(device).eval()

loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)
imgs, weather, masks = next(iter(loader))
with torch.no_grad():
    # Model outputs raw logits — apply sigmoid before visualisation
    preds = torch.sigmoid(
        model_B_loaded(imgs.to(device), weather.to(device))
    ).cpu()

fig = visualize_predictions(
    imgs, masks, preds, n_samples=4,
    save_path=f'{FIG_DIR}/03_multimodal_v2_predictions.png',
    title='Weather-Aware U-Net v2 Predictions',
)
plt.show()
print('Notebook 03 complete. Proceed to Notebook 04 for ablation study.')